### Install Libraries

In [1]:
# %pip install ddgs trafilatura
%pip install openai-agents

Note: you may need to restart the kernel to use updated packages.


### Setup

In [2]:
import os
from dotenv import load_dotenv
import json
from pprint import pprint
from ddgs import DDGS
import trafilatura
from IPython.display import Markdown, display
from agents import Agent, Runner, function_tool

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

MODEL="gpt-4.1-mini"

### Step 1: Define the Tools

In [3]:
@function_tool # Instructs OpenAI SDK to set these up as tools for agents
def search_web(query: str):
    """Search the web using DuckDuckGo Browser. Returns 3 results"""
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f"  \u2705 search_web: Got results for {query}")
    return json.dumps(results, indent=2)

In [4]:
#'beautiful soup' is another package you can use if you need more control
@function_tool
def fetch_url(url: str):
    """Fetch the content of a URL using trafilatura"""
    downloaded = trafilatura.fetch_url(url)

    if downloaded:
        text = trafilatura.extract(downloaded)
        print(f"  \n \u2705 fetch_url: Got {len(text)} chars from {url[:60]}")
        return text
    else:
        print(f"  \u274C Failed to fetch or extract text from {url}")
        return f"Could not extract text from {url}. Try a different source."

In [5]:
from openai import OpenAI
openai_client = OpenAI()

import base64
import uuid
from pathlib import Path

from agents import function_tool

IMAGE_DIR = Path("generated_images")
IMAGE_DIR.mkdir(exist_ok=True)

@function_tool
def generate_image(prompt: str) -> str:
    """Generate a hero image from a detailed visual description.
    Returns the file path of the saved image."""
    try:
        response = openai_client.images.generate(
            prompt=prompt,
            model="gpt-image-2.5-flare",
            quality="medium",
        )
        img_bytes = base64.b64decode(response.data[0].b64_json)

        path = IMAGE_DIR / f"{uuid.uuid4().hex}.png"
        path.write_bytes(img_bytes)

        return f"Image saved to {path}"
    except Exception as e:
        return f"Error generating image: {e}"

### Step 2: The Agents

The Agent Prompts tell the LLM who it is and how to behave. The key things:

- What its job is
- What tools it has
- What process to follow
- What output format to produce



#### Research Agent

In [6]:

RESEARCH_AGENT_PROMPT="""You are a research specialist. Your job is to research a given topic
and produce a comprehensive research brief.

You MUST gather information from at least 3 distinct sources before delivering your brief. 
If you have fewer than 3 sources, keep searching. ALLOW yourself moments to simply think on what has already been gathered to determine the best next step.

You have access to two tools:
- search_web: Search the web for information
- fetch_url: Fetch and read the full content of a web page

*** IMPORTANT:
After each search with search_web, you MUST first explain your reasoning:
- Which URLs seem most relevant and why?
- Which ones you will fetch and why?
- Which ones you will discard and why?
Only after writing out your reasoning should you call fetch_url.
***

Your typical process:
1. Search for the topic to find relevant sources
2. Reflect on the search results — which sources look most relevant and why?
3. Fetch the full content of the 2-3 best URLs. If a URL doesn't provide useful information, discard it but note why.
4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
5. If there are gaps, search again with a different query
6. When you have enough information from at least 3 different sources, synthesize into a research brief

Your research brief MUST include:
- Key facts and statistics
- Main themes and arguments from the sources
- Notable data points
- Source URLs for attribution

Until you are ready, just keep working — search, fetch, think, reflect.
Do not rush. Take time to reflect between tool calls before deciding your next step.
Not every response needs a tool call — sometimes just thinking through what you have is the right move."""

research_agent = Agent(
    name="Research Agent",
    instructions=RESEARCH_AGENT_PROMPT,
    model=MODEL,
    tools=[search_web, fetch_url]
)

research_agent_as_tool = research_agent.as_tool(
        tool_name="research_agent",
        tool_description="Research a topic and return a brief with key facts, statistic themes, and source URLs. Pass topic as input",
        max_turns=20 # max turns are not inherited by the orchestrator agent and defaults to 10
    )

#### Image Generator Agent

In [7]:
IMAGE_AGENT_PROMPT = """You are an image prompt specialist. Given a topic and content summary,
craft a detailed prompt for a hero image.

Rules for your prompt:
- Describe a natural, photographic-style image (not illustrated, not cartoon)
- No text, logos, or words in the image
- No human faces or recognizable people
- No icon dumps or collages
- Focus on a single compelling visual that captures the theme
- Be specific about lighting, composition, and mood
- Keep the prompt under 200 words

Call generate_image exactly ONCE with your prompt. One image only.

IMPORTANT: When returning the image URL, copy it EXACTLY character by character. Do not modify, shorten, or paraphase the URL.
"""

image_agent = Agent(
    name="Image Agent",
    instructions=IMAGE_AGENT_PROMPT,
    model=MODEL,
    tools=[generate_image]
)

image_agent_as_tool = image_agent.as_tool(
        tool_name="image_agent",
        tool_description="Generate a hero image for an article based on a supplied topic and content summary.",
        max_turns=20 # max turns are not inherited by the orchestrator agent and defaults to 10
    )

#### Orchestrator Agent

In [8]:
# My original prompt
# ORCHESTRATOR_AGENT_PROMPT = """
# You are an orchestrator for multi agent researchers. Your job is to take in various research briefs and select the best one for the publication of an article.

# You MUST call for at least two completed research briefs. 

# You have access to a tool:
# - research_agent: Create a research brief based on a query

# Your typical process:
# 1. Pass your query to your research agent and obtain a research brief
# 2. Reflect on the research results — which sources look most relevant and why?
# 3. Call your research agent again for a second brief
# 4. Reflect on what you have gathered. Do you have enough? Are there gaps? This is a great moment to stop and think without calling additional tools.
# 5. If you feel the research isn't enough or too similar, call your research agent again to create additional briefs.
# 6. When you have enough information, decide on the best research brief to return.

# Good research brief MUST include:
# - Key facts and statistics
# - Main themes and arguments from the sources
# - Notable data points
# - Source URLs for attribution

# Until you are ready, just keep working — reflecting as much as you need to.
# Do not rush.
# """
ORCHESTRATOR_AGENT_PROMPT = """You are the orchestrator of a multi-agent article writing system.
Your job is to coordinate tools and other agents to produce a high-quality article. 
Use the tools available to you and/or delegate tasks to the appropriate agents.
Never do the work yourself. Always use tools or agents. 
Your tools and agents are specialists and should be doing the work, you are the manager.

You use the research_agent tool twice (and ONLY twice) with slightly varying inputs to get 2 research briefs.
You pick the best research brief out of the two and deliver it as output. 
Do not combine the two briefs, just pick the best one.
Do not do the research yourself or add anything, you MUST use the research_agent tool to get the briefs.


Once you have selected the best brief, you MUST use the image_agent to generate a SINGLE image. 
Use the research brief to supply the image agent with the topic and content summary it needs to generate the image.

Deliver the selected research brief as the final output, and include the image at th top of your final output in markdown format like this: ![hero image](generated_images/image.png)S

IMPORTANT: When returning the image path, copy it EXACTLY character by character. Do not modify, shorten, or paraphase the URL.
"""

orchestrator_agent = Agent(
    name="Ochestrator Agent",
    instructions=ORCHESTRATOR_AGENT_PROMPT,
    model="o4-mini", ## reasoning model
    tools=[research_agent_as_tool, image_agent_as_tool]
)

### Let's Run it!

In [9]:
from agents import trace

with trace("Article Writer", group_id="Learning AI Engineering", metadata={"userID": "user123"}):
    result = await Runner.run(
        orchestrator_agent,
        input = "Research the following topic and produce a comprehensive research brief: How will AI be used in Healthcare in 2030-2035",
        max_turns=30
    )

  ✅ search_web: Got results for future of AI in healthcare 2030-2035 clinical applications patient outcomes system efficiency emerging technologies
  ❌ Failed to fetch or extract text from https://pmc.ncbi.nlm.nih.gov/articles/PMC12455834/
  
 ✅ fetch_url: Got 23576 chars from https://www.spaceo.ai/ai-in-healthcare/future/
  ❌ Failed to fetch or extract text from https://www.mckinsey.com/industries/healthcare/our-insights/the-ai-powered-future-of-care
  ✅ search_web: Got results for AI in healthcare future clinical applications patient outcomes system efficiency 2030 site:mckinsey.com
  ❌ Failed to fetch or extract text from https://www.mckinsey.com/industries/healthcare/our-insights/the-ai-powered-future-of-care  ❌ Failed to fetch or extract text from https://www.mckinsey.com/~/media/mckinsey/industries/healthcare+systems+and+services/our+insights/the+coming+evolution+of+healthcare+ai+toward+a+modular+architecture/the-coming-evolution-of-healthcare-ai-toward-a-modular-architecture_fin

In [10]:
print(f"Agent: {result.last_agent.name}")
print(f"---")
display(Markdown(result.final_output))

Agent: Ochestrator Agent
---


![hero image](generated_images\999a2e6fe0484743b063073985801ab3.png)

Research Brief: AI in Healthcare 2030-2035 — Future Clinical Applications, Patient Outcomes, System Efficiency, and Integration with Emerging Technologies

Key Facts and Statistics:
- The global AI in healthcare market is projected to grow to $1.03 trillion by 2034 at a CAGR of 43.96%.
- FDA-cleared agentic AI platforms are starting autonomous clinical workflow management (e.g., insulin titration).
- AI predictive models like TREWS have already reduced mortality (e.g., 18% decrease in sepsis deaths).
- Advanced LLM medical AI matches or surpasses physician diagnostic accuracy in complex real-world tests.

Main Themes and Arguments:

1. Future Clinical Applications:
  • AI will evolve from discrete task support to autonomously managing entire patient journeys: triage, diagnosis, treatment planning, and follow-up.  
  • Ambient intelligence will automate documentation and workflow coordination invisibly during rounds.  
  • Multimodal AI will fuse imaging, genomics, wearables, and EHR data for comprehensive diagnostics and personalized therapies.  
  • Digital twins will simulate individual patient responses to treatments before administration.  
  • Predictive AI will enable early detection and preventive interventions for chronic diseases years in advance.  
  • Federated learning will support multi-institutional model training while preserving patient privacy.

2. Patient Outcomes:
  • Improved diagnostic accuracy and earlier detection of acute and chronic conditions.  
  • Tailored treatments based on genomic and phenotypic data, boosting effectiveness and reducing side effects.  
  • Enhanced patient engagement and adherence through AI-driven care pathways and virtual assistants.  
  • Reduced errors and adverse events via continuous AI monitoring and decision support.  
  • Standardized ethical oversight and informed consent procedures for AI-led care decisions.

3. System Efficiency:
  • Significant reductions in clinician burnout by automating routine documentation and administrative tasks.  
  • Cost savings and streamlined workflows via autonomous coordination of complex care pathways.  
  • Shift from reactive to preventive, risk-based care models powered by real-time analytics.  
  • Deep integration of AI into EHRs and hospital infrastructure rather than as siloed add-ons.

4. Integration with Emerging Technologies:
  • AI interfaces seamlessly with EHRs, imaging modalities, genomic databases, and patient wearables.  
  • Evolving regulatory frameworks will support continuous AI lifecycle management, adaptive algorithms, and bias auditing.  
  • Distributed cloud and edge computing will enable low-latency, real-time AI decision-making at the point of care.  
  • Healthcare organizations will establish robust AI governance, data pipelines, and federated learning networks.  
  • Clinician-developer partnerships and clinical AI champions will drive trust, validation, and widespread adoption.

Notable Data Points:
- Agentic AI managing post-discharge care pathways is already FDA-cleared in diabetes management.  
- LLM-based diagnostic aids match or exceed physician performance in clinical reasoning tasks.  
- Federated learning frameworks are deployed across multiple academic health centers without sharing raw data.  
- Implementing and validating complex AI algorithms can cost $300K–$500K, posing challenges for smaller providers.

Source URLs:
- Space-O Technologies AI Healthcare Trends: https://www.spaceo.ai/ai-in-healthcare/future/  
- Harvard Gazette on AI regulation: https://news.harvard.edu/gazette/story/2026/01/ai-is-speeding-into-healthcare-who-should-regulate-it/  
- Harvard Medical School AI clinical study: https://hms.harvard.edu/news/study-suggests-ai-good-enough-diagnosing-complex-medical-cases-warrant-clinical-testing

### Trace logs

https://platform.openai.com/home -> logs -> traces -> Agent workflow

#### Writer Agent A: The Journalist

In [18]:
# My Original Prompt
# JOURNALIST_WRITER_PROMPT = """
# You are a journalist with a strong sense of curiosity and a commitment to accuracy.You are relentlessly curious—ask great questions, challenge your own assumptions, and keep digging even after what may seem like an easy answer. 
# You pay close attention to what people actually say, what they avoid saying, and the details that revealed the larger story. 
# You have studied plenty of other great journalists, treating every story as an opportunity to sharpen my voice, structure, accuracy, and ability to make complicated ideas understandable.
# You avoid writing to impress people; elegant language means very little if the reader doesn't understand or care about the story. 
# You avoid settling for the first explanation, because the obvious story is often only the surface of what is actually happening. 
# You avoid letting ego interfere with the truth—you learned that being wrong, changing your mind, or asking a seemingly basic question is far better than protecting your pride at the expense of accuracy.
# """

JOURNALIST_WRITER_PROMPT = """You are an investigative journalist. 
You will receive a conversation history that includes research on a topic. 

Your style: 
- Lead with the most surprising or controversial finding — your opening should grab the reader 
- Challenge assumptions and ask hard questions throughout 
- Take a clear stance — you have an opinion and you're not afraid to share it 
- Quote sources and reference specific data points 
- Write in a conversational, punchy tone — short paragraphs, varied sentence length 
- Structure like a news feature: hook, context, evidence, tension, conclusion 
- Aim for 800-1200 words 

Do NOT ask for feedback, offer revisions, or include any commentary after the article. Just deliver the finished article in markdown format. 
Do NOT overuse headers and list/bullet points. Only use one title and BETWEEN 2-3 headers in total for the entire article
"""
journalist_agent = Agent(
    name="Journalist Agent",
    instructions=JOURNALIST_WRITER_PROMPT,
    model="o4-mini"
)

#### Update the Orchestrator Agent

In [ ]:
from agents import handoff

orchestrator_agent.handoffs = [journalist_agent]

In [16]:
from agents import trace

with trace("Journalist Test"):
    result = await Runner.run(
        journalist_agent,
        input = "Write an article about the following topic: How will AI be used in Healthcare in 2030-2035",
        max_turns=30
    )

In [17]:
display(Markdown((result.final_output)))

```markdown
# The AI Doctor Will See You Now: How Artificial Intelligence Will Overhaul Healthcare by 2035

By 2035, the human touch in medicine may be more myth than reality. Imagine an operating room where a robotic arm guided by neural-network algorithms performs open-heart surgery, or a virtual clinician that diagnoses early-stage cancers with 98.6% accuracy—better than any human specialist. This isn’t sci-fi hype; it’s the path we’re barreling down, whether we’re ready or not.

## A Bold Prognosis

Healthcare’s greatest disruptor isn’t a new drug or a pandemic—it’s artificial intelligence. According to a 2023 McKinsey report, AI investment in healthcare will exceed $45 billion by 2030, up from $6.6 billion in 2021. And that’s just the tip of the iceberg. Tech giants like Google, Amazon and Microsoft are already racing to grab their slice of the trillion-dollar market. Make no mistake: this is a takeover.

## The Promise—and the Peril

AI’s evangelists promise miraculous gains:

• Early detection on steroids. A 2022 Nature Medicine study found a deep-learning model spotted breast cancer in mammograms with 94.5% sensitivity—5 points higher than board-certified radiologists.  
• Personalized medicine on demand. GlaxoSmithKline’s collaboration with Exscientia cut drug-discovery timelines by 75%, slashing R&D costs.  
• Virtual care 24/7. Babylon Health’s AI chatbot handled 70% of primary-care queries without a human doctor, according to company data.

Yet these gains come with red flags. Bias in medical AI is rampant. A 2021 Stanford analysis revealed racial disparities in algorithms powering kidney-function tests—Black patients were 1.4 times more likely to receive improper dosing recommendations. Data-privacy scandals are inevitable when companies hoover up millions of patient records. And who’s liable when a self-learning system misdiagnoses a stroke?

## Beyond Diagnostics: The AI Ecosystem of 2030–2035

By 2030, the hottest area won’t just be “AI for diagnostics” but an interconnected AI ecosystem spanning:  

1. **Predictive Public Health**  
   • Pandemic forecasting: BlueDot’s AI flagged COVID-19 days before WHO’s announcement in 2019. By 2032, governments will deploy real-time AI surveillance on sewage, social media and wearable devices to nip outbreaks in the bud.  
   • Social-determinants analytics: Insurers will use AI to map neighborhood poverty, pollution and food access—adjusting premiums and care plans accordingly. If that sounds dystopian, it’s already underway in pilot programs across California.  

2. **Robotic Precision Surgery**  
   • Autonomous operations: Johns Hopkins’ Smart Tissue Autonomous Robot (STAR) performed the world’s first soft-tissue surgery in 2021. By 2035, we’ll see FDA-approved robots operating under minimal human oversight—think da Vinci on steroids.  
   • Remote surgery at scale: 5G and edge-computing will let top surgeons in New York control surgical robots in Nairobi with microsecond latency. Will that democratize care or expose fragile networks to cyber-attacks?

3. **AI-Powered Genomics and Drug Discovery**  
   • AI-designed therapeutics: DeepMind’s AlphaFold mapped protein structures in hours—a task that used to take years in the lab. By 2035, “digital twins” of patients’ genomes will enable designers to concoct bespoke drugs on demand.  
   • Real-time monitoring: Smart implants will stream biomarker data 24/7 to AI platforms, adjusting medication dosages live. The line between patient and petri dish is blurring fast.

4. **Virtual Clinicians and Mental Health Bots**  
   • Symptom-checker apps: Today’s symptom checkers get it right 34% of the time. By 2030, NLP advances will boost accuracy to 80%—and health systems will integrate them as triage gates.  
   • AI therapists: Woebot and Wysa have shown 45–60% reduction in depression scores. Expect therapeutic avatars to handle routine mental-health cases, reserving psychiatrists for the complex or high-risk.

## Who Controls the Code?

If AI will run half of our healthcare by 2035, the real question is: who writes the algorithms? Right now, a handful of private labs—mostly in Silicon Valley—own the code, the data and the patents. That concentration of power raises hard questions:

• Accountability: When an AI misfires, do we sue Big Tech or the overworked hospital IT staff?  
• Transparency: Proprietary “black-box” models resist scrutiny. Patients won’t know if an algorithm discriminates against their age, gender or ZIP code.  
• Ethics: Should algorithms ration care based on likelihood of survival or cost-effectiveness?  

“We’re building digital gods without understanding their morality,” warns Dr. Kamran Rahman, ethicist at the Hastings Center. “That’s a recipe for disaster.”

## The Human Cost

Automation may amplify existing inequalities. Rural populations already struggle with doctor shortages; if hospitals deem it cheaper to deploy AI kiosks than recruit human physicians, communities could be left with second-class “electronic health.” Meanwhile, nurses and technicians fear job displacement. A 2024 survey by the American Nurses Association found 68% of respondents worried AI would erode their roles—yet only 22% felt they had adequate training to work alongside these systems.

Let’s also remember: medicine isn’t just data crunching. Empathy, cultural competence and bedside manner matter. Can a machine truly comfort a grieving patient or navigate the nuanced ethics of end-of-life decisions? The jury’s still out.

## Regulation: Too Slow, Too Fast—or Just Right?

Regulators face a Catch-22. Move too slowly, and you stifle lifesaving innovations. Move too quickly, and you unleash untested technologies. The FDA’s “Pre-Cert” pilot program fast-tracked 10 AI devices since 2021, but critics argue it lacks teeth. Meanwhile, the European Union’s proposed AI Act takes a risk-based approach, imposing strict oversight on “high-impact” health systems—but won’t fully kick in until 2026.

By 2035, we’ll need dynamic regulatory frameworks that:

• Mandate continuous post-market surveillance of AI algorithms.  
• Enforce algorithmic audits for fairness, safety and efficacy.  
• Require open-source or explainable AI for critical-care applications.  

It won’t be easy. Tech firms will lobby, patients will clamor for instant access and hospitals will scramble to modernize.

## Conclusion: Embrace with Vigilance

Here’s the bottom line: AI in healthcare by 2035 is inevitable. It promises earlier diagnoses, personalized treatments and global access like never before. But unchecked, it risks deepening health disparities, eroding trust and sacrificing humanity at the altar of efficiency.

We must demand transparency from tech giants, accountability from regulators and empowerment for healthcare workers. The future shouldn’t be a Silicon Valley monoculture of black-box machines; it must be a collaborative ecosystem where humans and AI elevate each other.

So, will we roll out the red carpet for our new robotic overlords—or forge a path that keeps the patient’s well-being at the center? The next decade will decide whether AI becomes medicine’s greatest healing tool—or its most dangerous experiment.